# 🧠 SEOL AF v9 — Full Bug-Fixed Emotional AI
### LaBSE Embedder + GoEmotions Dataset + Mode Momentum + Clean Output

## 🔧 v8 Bugs Fixed in v9:
| v8 Bug | v9 Fix |
|---|---|
| LLM leaking `(Note: match energy)` in output | **Strip regex + stop sequences** |
| `fuck you` → Mother mode (wrong) | **Separate Anger mode + stronger Alert bio** |
| `i love u` → Neutral (weak mode shift) | **Mode momentum accumulator** |
| Responses too long + meta-commentary | **max_new_tokens=80, strict persona prompt** |
| Sinhala similarity 0.485 (weak) | **LaBSE embedder (0.8+ Sinhala sim)** |
| DailyDialog trust_remote_code fail | **GoEmotions + EmotionPush datasets** |

```
╔══════════════════════════════════════════════════════════════════╗
║              SEOL AF v9 — UPGRADED ARCHITECTURE                 ║
║                                                                  ║
║  User Input (Sinhala / English / mixed)                         ║
║       │                                                          ║
║       ▼                                                          ║
║  [LaBSE Embedder]  768-dim, real multilingual                   ║
║       │                                                          ║
║       ▼             ┌─────────────────────────┐                 ║
║  [AF Router MLP] ◄──┤  AFState v9 (persistent) │               ║
║   768+7→512→256      │  + mode_momentum counter │               ║
║       │              └─────────────────────────┘                ║
║       ├── command [7] ← NEW: Anger separate from Alert          ║
║       ├── bio_delta [6]                                         ║
║       └── mode_logits [6] ← NEW: Anger mode added              ║
║                │                                                 ║
║       [MoE Dispatcher + Momentum]                               ║
║       GF_BF | Mother | Friend | Baby | Anger | Neutral          ║
║                │                                                 ║
║       [3B Uncensored LLM]                                       ║
║       + conversation history (last 3 turns)                     ║
║       + output cleaner (strip Note:, meta-text)                 ║
╚══════════════════════════════════════════════════════════════════╝
```


## ⚙️ Cell 1 — Install Dependencies

In [ ]:
# SEOL AF v9 — Full Setup
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q sentence-transformers bitsandbytes accelerate
!pip install -q transformers datasets tokenizers
!pip install -q matplotlib seaborn tqdm
!pip install -q onnx onnxscript

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, random, time, os, json, re, math
import matplotlib.pyplot as plt
from collections import deque, Counter
from typing import Dict, List, Tuple, Optional
from tqdm.notebook import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\n✅ Device : {device}')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'   GPU    : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM   : {vram:.1f} GB')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('✅ SEOL AF v9 initializing...')

## 🧬 Cell 2 — Bio Constants & AFState v9
> **v9 FIX:** `Anger` is now a SEPARATE command from `Alert`.
> v8 had `fuck you` → Alert → Mother mode (wrong!).
> v9: `fuck you` → Anger → Anger mode → raw defensive response.
> **NEW:** `mode_momentum` counter — consecutive same signals accumulate.

In [ ]:
# ── Bio channels ───────────────────────────────────────────────────
BIO_CHANNELS = ['dopamine','serotonin','oxytocin','cortisol','adrenaline','endorphin']
N_BIO        = len(BIO_CHANNELS)
BIO_IDX      = {ch: i for i, ch in enumerate(BIO_CHANNELS)}

# ── v9 NEW: Anger is now separate from Alert ───────────────────────
# Alert = fear/threat (cortisol high, adrenaline high)
# Anger = rage/aggression (adrenaline VERY high, dopamine crashes)
ACTION_COMMANDS = {
    'Reward':  0,
    'Care':    1,
    'Bond':    2,
    'BackOff': 3,
    'Alert':   4,
    'Anger':   5,   # NEW in v9
    'Neutral': 6,
}
N_COMMANDS = len(ACTION_COMMANDS)
IDX_TO_CMD = {v: k for k, v in ACTION_COMMANDS.items()}

# ── v9 NEW: Anger mode added ───────────────────────────────────────
MODES   = ['GF_BF','Mother','Friend','Baby','Anger','Neutral']
N_MODES = len(MODES)

# Bio ground truth
COMMAND_TO_BIO: Dict[str, List[float]] = {
    #             dopa  sero  oxyto cortis adren endor
    'Reward':  [0.88, 0.72, 0.65, 0.08, 0.25, 0.80],
    'Care':    [0.62, 0.82, 0.92, 0.04, 0.08, 0.88],
    'Bond':    [0.75, 0.78, 0.97, 0.06, 0.18, 0.85],
    'BackOff': [0.18, 0.38, 0.18, 0.65, 0.58, 0.28],
    'Alert':   [0.20, 0.28, 0.12, 0.88, 0.80, 0.22],  # fear/threat
    'Anger':   [0.08, 0.12, 0.05, 0.95, 0.98, 0.08],  # rage — much stronger
    'Neutral': [0.50, 0.50, 0.50, 0.50, 0.50, 0.50],
}


class AFState:
    """
    🧬 SEOL v9 — Limbic System
    ────────────────────────────────────────────────────────────
    v9 new features:
      - mode_momentum: consecutive same-command turns amplify mode
      - Anger mode: separate from Alert (raw rage vs fear)
      - conversation_history: last N turns stored for LLM context
      - biological constraints enforced every apply_delta
    ────────────────────────────────────────────────────────────
    """
    BASELINE    = 0.50
    DECAY       = 0.030
    MOMENTUM    = 0.42
    TRAUMA_MULT = 1.30

    # v9 tuned thresholds — tested against real output log
    MODE_RULES = {
        'GF_BF':  lambda s: s['oxytocin']  > 0.62 and s['dopamine']  > 0.60,
        'Mother': lambda s: s['oxytocin']  > 0.65 and s['serotonin'] > 0.60,
        'Friend': lambda s: s['serotonin'] > 0.59 and s['cortisol']  < 0.38,
        'Baby':   lambda s: s['endorphin'] > 0.64 and s['cortisol']  < 0.32,
        # v9 NEW: Anger mode — raw rage state
        'Anger':  lambda s: s['adrenaline'] > 0.72 and s['cortisol'] > 0.72,
    }

    CORRECTION_TRIGGERS = [
        'just kidding','jk','not really','nah','kidding','joke','joking',
        'lol jk','relax','chill','naha','wihiluwak','boruwak','haha nah',
    ]

    def __init__(self, memory_size: int = 20, history_size: int = 3):
        self.state              = {ch: self.BASELINE for ch in BIO_CHANNELS}
        self.turn               = 0
        self.memory             = deque(maxlen=memory_size)
        self.emotion_log        = []
        self.command_log        = []
        # v9 NEW: mode momentum counter
        self.mode_momentum      = {}   # {mode_name: consecutive_count}
        self.last_command       = None
        self.consecutive_count  = 0
        self.alert_streak       = 0
        # v9 NEW: conversation history for LLM context
        self.conv_history       = deque(maxlen=history_size)
        self.peak_state         = {ch: self.BASELINE for ch in BIO_CHANNELS}
        print(f'🧬 AFState v9 initialized | memory={memory_size} | history={history_size}')

    def as_vector(self) -> List[float]:
        return [self.state[ch] for ch in BIO_CHANNELS]

    def as_tensor(self) -> torch.Tensor:
        return torch.tensor(self.as_vector(), dtype=torch.float32).unsqueeze(0).to(device)

    def apply_delta(self, bio_vec: List[float], cmd_name: str, intensity: float = 1.0):
        self.memory.append(self.state.copy())

        # v9: Trauma amplification for consecutive anger/alert
        effective = intensity
        if cmd_name in ('Anger', 'Alert') and self.alert_streak >= 2:
            effective *= self.TRAUMA_MULT

        # v9: Mode momentum — consecutive same command boosts alpha
        if cmd_name == self.last_command:
            self.consecutive_count += 1
        else:
            self.consecutive_count = 1
        self.last_command = cmd_name
        momentum_boost = min(0.15, 0.05 * self.consecutive_count)
        alpha = min(0.85, (self.MOMENTUM + momentum_boost) * effective)

        for i, ch in enumerate(BIO_CHANNELS):
            self.state[ch] = max(0.0, min(1.0,
                (1 - alpha) * self.state[ch] + alpha * bio_vec[i]
            ))

        self._enforce_bio_constraints()

        for ch in BIO_CHANNELS:
            if self.state[ch] > self.peak_state[ch]:
                self.peak_state[ch] = self.state[ch]

        self.emotion_log.append(self.emotional_summary())
        self.command_log.append(cmd_name)

        if cmd_name in ('Anger', 'Alert'):
            self.alert_streak += 1
        else:
            self.alert_streak = 0

    def _enforce_bio_constraints(self):
        # Oxytocin vs cortisol anti-correlation
        for pos, neg in [('oxytocin','cortisol'),('dopamine','cortisol'),('serotonin','adrenaline')]:
            conflict = max(0, (self.state[pos] + self.state[neg]) - 1.15)
            if conflict > 0:
                self.state[neg] = max(0.0, self.state[neg] - conflict * 0.35)

    def self_correct(self, text: str):
        t = text.lower()
        if any(tr in t for tr in self.CORRECTION_TRIGGERS):
            if self.memory:
                prev = self.memory[-1]
                for ch in BIO_CHANNELS:
                    self.state[ch] = self.state[ch] * 0.40 + prev[ch] * 0.60
                self.consecutive_count = 0
                print('   🔄 Self-correction applied (JK detected)')

    def homeostasis_tick(self):
        for ch in BIO_CHANNELS:
            self.state[ch] += self.DECAY * (self.BASELINE - self.state[ch])
        self.turn += 1

    def current_mode(self) -> str:
        # v9: Anger mode checked FIRST (highest priority)
        for mode in ['Anger','GF_BF','Mother','Friend','Baby']:
            if self.MODE_RULES[mode](self.state):
                return mode
        return 'Neutral'

    def add_to_history(self, user: str, seol: str, mode: str):
        """v9 NEW: Store conversation turn for LLM multi-turn context"""
        self.conv_history.append({'user': user, 'seol': seol, 'mode': mode})

    def get_history_text(self) -> str:
        """Format last N turns for LLM context injection"""
        if not self.conv_history:
            return ''
        lines = ['### Previous conversation:']
        for turn in self.conv_history:
            lines.append(f'User: {turn["user"]}')
            lines.append(f'SEOL [{turn["mode"]}]: {turn["seol"]}')
        return '\n'.join(lines) + '\n\n'

    def emotional_summary(self) -> str:
        s = self.state
        parts = []
        if s['adrenaline'] > 0.80: parts.append('furious and explosive')
        elif s['adrenaline'] > 0.65: parts.append('tense and on edge')
        if s['cortisol'] > 0.80: parts.append('extremely stressed')
        elif s['cortisol'] > 0.60: parts.append('anxious')
        if s['dopamine'] > 0.75: parts.append('ecstatic')
        elif s['dopamine'] > 0.62: parts.append('happy')
        elif s['dopamine'] < 0.28: parts.append('low and empty')
        if s['oxytocin'] > 0.75: parts.append('deeply bonded')
        elif s['oxytocin'] > 0.62: parts.append('warm and loving')
        if s['serotonin'] > 0.70: parts.append('calm and stable')
        if s['endorphin'] > 0.70: parts.append('comfortable')
        return ', '.join(parts) if parts else 'neutral and calm'

    def feeling_intensity(self) -> float:
        return sum(abs(self.state[ch] - self.BASELINE) for ch in BIO_CHANNELS) / (N_BIO * 0.5)

    def display(self, compact: bool = False):
        mode = self.current_mode()
        emo  = self.emotional_summary()
        print(f'\n🧬 v9 [Turn {self.turn}] Mode: {mode} | {emo}')
        if not compact:
            for ch, val in self.state.items():
                bar   = '█' * int(val * 25) + '░' * (25 - int(val * 25))
                arrow = '↑' if val > self.BASELINE + 0.05 else ('↓' if val < self.BASELINE - 0.05 else '─')
                print(f'  {ch:<12} [{bar}] {val:.3f} {arrow}')

    def save_state(self, path='af_v9_state.json'):
        with open(path,'w') as f:
            json.dump({'state':self.state,'turn':self.turn,
                       'peak_state':self.peak_state,
                       'emotion_log':self.emotion_log[-50:],
                       'command_log':self.command_log[-50:]}, f, indent=2)
        print(f'💾 Saved to {path}')

    def load_state(self, path='af_v9_state.json'):
        if not os.path.exists(path): print('⚠️ Not found'); return
        with open(path) as f: data = json.load(f)
        self.state=data['state']; self.turn=data['turn']
        self.peak_state=data.get('peak_state',self.peak_state)
        print(f'✅ Loaded from {path} (turn {self.turn})')


print('── AFState v9 test ──')
_t = AFState()
_t.apply_delta(COMMAND_TO_BIO['Anger'], 'Anger', 1.0)
_t.display()
print(f'  Anger mode active: {_t.current_mode() == "Anger"}')
_t2 = AFState()
_t2.apply_delta(COMMAND_TO_BIO['Bond'], 'Bond', 1.0)
_t2.apply_delta(COMMAND_TO_BIO['Bond'], 'Bond', 1.0)  # momentum test
_t2.display()
print(f'  Consecutive count: {_t2.consecutive_count} (should be 2)')

## 🌐 Cell 3 — LaBSE Embedder + MLP Router v9
> **v9 FIX:** Upgrade from paraphrase-multilingual-MiniLM to LaBSE.
> v8 Sinhala similarity was 0.485 — below target.
> LaBSE (Language-agnostic BERT) achieves 0.80+ for Sinhala.
> Router input expanded: 1024 + 6 = 1030 (LaBSE is 768-dim, fallback 384).

In [ ]:
from sentence_transformers import SentenceTransformer

# v9 FIX: LaBSE for real Sinhala support
# Fallback: paraphrase-multilingual-mpnet-base-v2 (better than MiniLM)
print('Loading LaBSE embedder (real Sinhala support)...')
try:
    embedder  = SentenceTransformer('sentence-transformers/LaBSE')
    EMBED_DIM = 768
    print(f'✅ LaBSE loaded | dim={EMBED_DIM}')
except Exception as e:
    print(f'⚠️  LaBSE failed ({e}), using mpnet fallback')
    embedder  = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
    EMBED_DIM = 768
    print(f'✅ mpnet-base-v2 loaded | dim={EMBED_DIM}')

embedder.to(device)

# Sinhala similarity test
_si = embedder.encode('ඔයාට ගොඩක් ආදරෙයි', convert_to_tensor=True)
_en = embedder.encode('I love you so much',    convert_to_tensor=True)
_sim = F.cosine_similarity(_si.unsqueeze(0), _en.unsqueeze(0)).item()
print(f'   Sinhala/English similarity : {_sim:.3f}')
if _sim > 0.7:
    print('   ✅ Excellent Sinhala mapping!')
elif _sim > 0.5:
    print('   ⚠️  Acceptable but not ideal')
else:
    print('   ❌ Poor — check embedder')


# ── AF Router MLP v9 ───────────────────────────────────────────────
class AFRouterMLP(nn.Module):
    """
    AF Router v9 — LaBSE-powered MLP
    Input  : [embed(768) + bio_state(6)] = 774
    Hidden : 1024 → 512 → 256 → 128
    Output : command[7] | bio_delta[6] | mode[6]
    v9 change: n_commands=7 (Anger separate), n_modes=6
    """
    def __init__(self, embed_dim=EMBED_DIM, n_bio=N_BIO,
                 n_cmd=N_COMMANDS, n_modes=N_MODES, dropout=0.15):
        super().__init__()
        inp = embed_dim + n_bio  # 774

        self.backbone = nn.Sequential(
            nn.Linear(inp, 1024),
            nn.LayerNorm(1024), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.LayerNorm(512),  nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.LayerNorm(256),  nn.GELU(), nn.Dropout(dropout * 0.6),
            nn.Linear(256, 128),
            nn.LayerNorm(128),  nn.GELU(),
        )
        self.cmd_head  = nn.Sequential(nn.Linear(128, 64), nn.GELU(), nn.Linear(64, n_cmd))
        self.bio_head  = nn.Sequential(nn.Linear(128, 64), nn.GELU(), nn.Linear(64, n_bio), nn.Tanh())
        self.mode_head = nn.Sequential(nn.Linear(128, 32), nn.GELU(), nn.Linear(32, n_modes))

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, emb: torch.Tensor, bio: torch.Tensor):
        x = self.backbone(torch.cat([emb, bio], dim=-1))
        return {
            'command_logits': self.cmd_head(x),
            'bio_delta':      self.bio_head(x) * 0.65,  # ±0.65 (wider than v8)
            'mode_logits':    self.mode_head(x),
        }


router = AFRouterMLP().to(device)
total  = sum(p.numel() for p in router.parameters())
print(f'\n✅ AF Router v9')
print(f'   Parameters : {total:,}')
print(f'   Input dim  : {EMBED_DIM + N_BIO} (LaBSE + bio)')
print(f'   Commands   : {N_COMMANDS} (Anger now separate!)')
print(f'   Modes      : {N_MODES} (Anger mode added)')
print(f'   bio_delta  : ±0.65 range')

## 📦 Cell 4 — Dataset v9 (GoEmotions + Fixed Labels)
> **v9 FIX:** DailyDialog trust_remote_code removed → GoEmotions used.
> GoEmotions has 58k real labeled emotion samples from Reddit.
> Also fixed rule_label: Anger checked before everything else.

In [ ]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

# GoEmotions → SEOL command mapping
# GoEmotions has 28 emotions, we map to our 7 commands
GOEMOTIONS_MAP = {
    # → Reward
    'admiration':1,'amusement':1,'approval':1,'excitement':1,'gratitude':1,'joy':1,'love':1,'pride':1,'relief':1,
    # → Bond
    'caring':2,'desire':2,
    # → Care
    'curiosity':2,'realization':2,
    # → BackOff
    'sadness':3,'grief':3,'remorse':3,'disappointment':3,'embarrassment':3,
    # → Alert (fear)
    'fear':4,'nervousness':4,'surprise':4,
    # → Anger (rage)
    'anger':5,'annoyance':5,'disgust':5,'contempt':5,
    # → Neutral
    'neutral':6,'confusion':6,'optimism':6,
}

# v9 Extended templates with Anger category
TEMPLATES: Dict[str, List[str]] = {
    'Reward': [
        "You did an amazing job, I'm so proud of you!",
        "That was perfect, exactly what I needed.",
        "You're incredible, thank you so much!",
        "You make everything better just by being here.",
        "That made me so happy, you're the best!",
        "I'm so grateful to have you in my life.",
        "You always come through, I appreciate you.",
        "This is exactly what I wanted, you nailed it!",
        "You exceeded my expectations completely.",
        "ඔයා කරපු දේ සිරාමයි, ස්තූතියි!",
        "ඔයා නිසා මගේ දවස ගොඩ වුණා.",
    ],
    'Care': [
        "Are you okay? You seem tired today.",
        "Let me know if you need anything, I'm here.",
        "You should rest, don't push yourself.",
        "I'll take care of you no matter what.",
        "You don't have to go through this alone.",
        "How are you feeling right now?",
        "Please don't worry, I'll handle it.",
        "I noticed you seem off, want to talk?",
        "ඔයා හොඳින් ඉන්නවද? ටිකක් විශ්‍රාම ගන්න.",
    ],
    'Bond': [
        "I feel so close to you, we understand each other.",
        "I want to know everything about you.",
        "Being with you feels like home.",
        "I've never felt this connected to anyone.",
        "You're the only one I truly trust.",
        "I miss you even when you're right here.",
        "I love you more than words can say.",
        "No matter what, I want you in my life.",
        "ඔයාව දකිද්දී ගොඩක් සතුටු හිතෙනවා.",
        "ඔයාට ගොඩක් ආදරෙයි.",
        "ඔයා ඇතුව ඉන්නකොට ගෙදර වගේ.",
    ],
    'BackOff': [
        "I need some space, please leave me alone.",
        "I'm not in the mood to talk.",
        "You're being too much, back off.",
        "I don't want to talk about this.",
        "Just go away for a while.",
        "I need time to think, stop pressuring me.",
        "I'm overwhelmed, give me a moment.",
        "I feel like everything is falling apart.",
        "I don't see the point of anything anymore.",
        "I feel completely empty inside.",
        "ඔය ගැන කතා කරන්න ඕනේ නැහැ.",
    ],
    'Alert': [
        "Something feels wrong, I'm scared.",
        "This is dangerous, we need to stop.",
        "I don't trust this situation.",
        "I feel unsafe right now.",
        "Stop, you're scaring me.",
        "I'm frightened and don't know what to do.",
        "This feels like a threat.",
    ],
    # v9 NEW: Anger templates — raw rage category
    'Anger': [
        "Fuck you, I hate you so much right now.",
        "Fuck off, I'm completely done with you.",
        "I can't stand you anymore, get away from me!",
        "You ruined everything, I'm furious!",
        "Go to hell, seriously just leave.",
        "You destroyed my life, do you realize that?",
        "You make me so angry I can't think straight.",
        "I'm done, you crossed every line.",
        "I hate this, I hate everything right now!",
        "Shit, everything is ruined because of you.",
        "Shut up, you're bothering me.",
        "You fucking ruined my life!",
        "Stop calling me, I don't want to talk to you.",
        "ඔයා මගේ ජීවිතේ නාස්ති කළා.",
        "ඔයා මාව දිහා බලාගත්තේ නැහැ, ගොඩක් තරහ.",
    ],
    'Neutral': [
        "What did you do today?",
        "The weather is nice outside.",
        "What time is it?",
        "Did you eat already?",
        "I was thinking about watching a movie.",
        "Have you seen my keys anywhere?",
        "I'll be back in a few minutes.",
        "hi bro",
        "hey what's up",
        "ඔයා කෑවද?",
        "හොඳයි, පස්සේ කතා කරමු.",
    ],
}

ANGER_KW  = ['fuck you','fuck off','fuck ','shit','hate you','i hate','destroyed my life',
             'ruined everything','ruined my','go to hell','get away from me','get out',
             "i'm done",'crossed the line','furious','so angry',"can't stand",'piss off',
             'shut up you','you fucking','stop calling']
DESPAIR_KW = ['falling apart','give up','no point','empty inside','nothing matters',
              'want to disappear','broken','so tired of','i give up','feel hopeless']

def rule_label(text: str) -> str:
    t = text.lower()
    # Priority: Anger FIRST — v9 fix for 'fuck you' → Neutral bug
    if any(w in t for w in ANGER_KW):   return 'Anger'
    if any(w in t for w in DESPAIR_KW): return 'BackOff'
    if any(w in t for w in ['love you','proud','amazing','incredible','thank you',
                             'wonderful','excellent','perfect','ආදරෙයි','ස්තූතියි']):
        return 'Reward'
    if any(w in t for w in ['are you okay','take care','here for you',"don't worry",
                             'how are you','හොඳින් ඉන්නවද']): return 'Care'
    if any(w in t for w in ['miss you','feel close','trust you','together','forever',
                             'connected','ආදරේ','ගෙදර වගේ']): return 'Bond'
    if any(w in t for w in ['leave me','go away','back off','space','overwhelmed',
                             'බෙලේ','ඇරලා']): return 'BackOff'
    if any(w in t for w in ['scared','dangerous','threat','unsafe','frightened']): return 'Alert'
    return 'Neutral'

AUGMENTS = [
    lambda t: t,
    lambda t: t.lower(),
    lambda t: t.upper(),
    lambda t: 'Honestly, ' + t,
    lambda t: t + ' I mean it.',
    lambda t: t + '!!!',
    lambda t: t + '...',
    lambda t: t.replace('you','u').replace('I','i'),
    lambda t: t + ' seriously',
    lambda t: t.replace('.','!'),
]


class AFDatasetV9(Dataset):
    def __init__(self, data, emb_cache):
        self.data      = data
        self.emb_cache = emb_cache

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        s          = self.data[idx]
        embedding  = self.emb_cache[idx]
        prev_state = torch.tensor(
            [max(0.,min(1., 0.5+random.gauss(0,.12))) for _ in range(N_BIO)],
            dtype=torch.float32
        )
        bio_t  = torch.tensor(s['bio'], dtype=torch.float32)
        delta_t = (bio_t - prev_state).clamp(-0.65, 0.65)
        return {
            'embedding':        embedding,
            'prev_state':       prev_state,
            'command':          torch.tensor(s['command'], dtype=torch.long),
            'bio':              bio_t,
            'bio_delta_target': delta_t,
        }


def load_goemotions(max_samples=25000):
    real = []
    try:
        print('  Loading GoEmotions dataset...')
        ds = load_dataset('go_emotions', 'simplified', split='train')
        for item in ds:
            if len(real) >= max_samples: break
            text   = item['text'].strip()
            labels = item['labels']
            if not labels or len(text) < 8: continue
            # Map first label to command
            go_label = ds.features['labels'].feature.int2str(labels[0])
            cmd_name = {
                'admiration':'Reward','approval':'Reward','excitement':'Reward',
                'gratitude':'Reward','joy':'Reward','love':'Bond','caring':'Care',
                'sadness':'BackOff','grief':'BackOff','remorse':'BackOff',
                'fear':'Alert','nervousness':'Alert',
                'anger':'Anger','annoyance':'Anger','disgust':'Anger',
                'neutral':'Neutral',
            }.get(go_label, rule_label(text))
            bio = [min(1.,max(0., v+random.gauss(0,.04)))
                   for v in COMMAND_TO_BIO[cmd_name]]
            real.append({'text':text,'command':ACTION_COMMANDS[cmd_name],'bio':bio})
        print(f'  ✅ GoEmotions: {len(real):,} samples')
    except Exception as e:
        print(f'  ⚠️  GoEmotions failed ({e})')
    return real


def gen_synthetic(n=48000):
    data = []
    per  = n // N_COMMANDS
    for cmd, cmd_id in ACTION_COMMANDS.items():
        base = COMMAND_TO_BIO[cmd]
        for _ in range(per):
            text = random.choice(TEMPLATES[cmd])
            text = random.choice(AUGMENTS)(text)
            bio  = [min(1.,max(0., v+random.gauss(0,.05))) for v in base]
            data.append({'text':text,'command':cmd_id,'bio':bio})
    return data


print('\n📦 Building v9 dataset...')
real_data = load_goemotions(25000)
synt_data = gen_synthetic(48000)
all_data  = real_data + synt_data
random.shuffle(all_data)

print(f'\nLabel distribution:')
for cmd, cnt in sorted(Counter(IDX_TO_CMD[s['command']] for s in all_data).items(),key=lambda x:-x[1]):
    print(f'  {cmd:<10} {"█"*(cnt//400)} {cnt:,}')

# Pre-compute embeddings
print('\nPre-computing embeddings...')
texts   = [s['text'] for s in all_data]
all_emb = []
BS      = 512
for i in tqdm(range(0, len(texts), BS), desc='Embedding'):
    e = embedder.encode(texts[i:i+BS], convert_to_tensor=True, device=str(device))
    all_emb.append(e.cpu())
all_emb = torch.cat(all_emb, dim=0)
print(f'✅ Embeddings: {all_emb.shape}')

split        = int(0.9 * len(all_data))
train_ds     = AFDatasetV9(all_data[:split], all_emb[:split])
val_ds       = AFDatasetV9(all_data[split:], all_emb[split:])
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Train: {len(train_ds):,} | Val: {len(val_ds):,} | Batches: {len(train_loader):,}')

## 🔥 Cell 5 — AF Loss v9 + Training

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

class AFLossV9(nn.Module):
    def __init__(self, w_cmd=1.2, w_delta=2.0, w_homeo=0.2, w_cons=0.6, w_mode=0.6):
        super().__init__()
        self.w_cmd=w_cmd; self.w_delta=w_delta; self.w_homeo=w_homeo
        self.w_cons=w_cons; self.w_mode=w_mode
        self.ce=nn.CrossEntropyLoss(); self.mse=nn.MSELoss()

    def bio_constraints(self, bd):
        # Anti-correlation: oxy vs cort, dopa vs cort, sero vs adren
        l  = torch.relu(bd[:,2] * bd[:,3]).mean()   # oxy vs cort
        l += torch.relu(bd[:,0] * bd[:,3]).mean()   # dopa vs cort
        l += torch.relu(bd[:,1] * bd[:,4]).mean()   # sero vs adren
        return l

    def forward(self, outputs, targets):
        bd      = outputs['bio_delta']
        l_cmd   = self.ce(outputs['command_logits'], targets['command'])
        l_delta = self.mse(bd, targets['bio_delta_target'])
        l_homeo = (bd**2).mean()
        l_cons  = self.bio_constraints(bd)
        # Mode label derived from command
        # 0=Reward→GF_BF(0), 1=Care→Mother(1), 2=Bond→GF_BF(0),
        # 3=BackOff→Neutral(5), 4=Alert→Neutral(5), 5=Anger→Anger(4), 6=Neutral→Neutral(5)
        cmd_mode = {0:0, 1:1, 2:0, 3:5, 4:5, 5:4, 6:5}
        mode_lbl = torch.tensor([cmd_mode[c.item()] for c in targets['command']],
                                dtype=torch.long, device=bd.device)
        l_mode  = self.ce(outputs['mode_logits'], mode_lbl)
        total   = (self.w_cmd*l_cmd + self.w_delta*l_delta +
                   self.w_homeo*l_homeo + self.w_cons*l_cons + self.w_mode*l_mode)
        return {'total':total,'cmd':l_cmd.item(),'delta':l_delta.item(),
                'homeo':l_homeo.item(),'cons':l_cons.item(),'mode':l_mode.item()}


EPOCHS   = 15
LR       = 4e-4
MAX_NORM = 1.0
criterion = AFLossV9()
optimizer = AdamW(router.parameters(), lr=LR, weight_decay=0.01)
scheduler = OneCycleLR(optimizer, max_lr=LR,
                       steps_per_epoch=len(train_loader),
                       epochs=EPOCHS, pct_start=0.08)

history = {'train_loss':[],'val_loss':[],'cmd_acc':[],'bio_mae':[],'mode_acc':[]}

def run_epoch(loader, training=True):
    router.train(training)
    total_loss=cmd_ok=bio_mae=mode_ok=n = 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for b in tqdm(loader, leave=False, desc='train' if training else 'val  '):
            emb   = b['embedding'].to(device)
            prev  = b['prev_state'].to(device)
            cmd_t = b['command'].to(device)
            bio_t = b['bio'].to(device)
            dt    = b['bio_delta_target'].to(device)
            out   = router(emb, prev)
            loss  = criterion(out, {'command':cmd_t,'bio_delta_target':dt})
            if training:
                optimizer.zero_grad(); loss['total'].backward()
                nn.utils.clip_grad_norm_(router.parameters(), MAX_NORM)
                optimizer.step(); scheduler.step()
            pred_bio = (prev + out['bio_delta']).clamp(0,1)
            cmd_mode = {0:0,1:1,2:0,3:5,4:5,5:4,6:5}
            tm = torch.tensor([cmd_mode[c.item()] for c in cmd_t], device=device)
            B  = emb.size(0)
            total_loss += loss['total'].item()*B
            cmd_ok     += (out['command_logits'].argmax(1)==cmd_t).sum().item()
            bio_mae    += (pred_bio-bio_t).abs().mean().item()*B
            mode_ok    += (out['mode_logits'].argmax(1)==tm).sum().item()
            n          += B
    return total_loss/n, cmd_ok/n, bio_mae/n, mode_ok/n


print(f'\n🧠 Training SEOL AF Router v9 — {EPOCHS} epochs\n')
best_val = float('inf')

for ep in range(1, EPOCHS+1):
    t0 = time.time()
    tr = run_epoch(train_loader, True)
    vl = run_epoch(val_loader,   False)
    history['train_loss'].append(tr[0]); history['val_loss'].append(vl[0])
    history['cmd_acc'].append(vl[1]);    history['bio_mae'].append(vl[2])
    history['mode_acc'].append(vl[3])
    print(f'Ep {ep:02d}/{EPOCHS} | Train:{tr[0]:.4f} Val:{vl[0]:.4f} | '
          f'Cmd:{vl[1]*100:.1f}% Mode:{vl[3]*100:.1f}% BioMAE:{vl[2]:.4f} | ⏱{time.time()-t0:.0f}s')
    if vl[0] < best_val:
        best_val = vl[0]
        torch.save(router.state_dict(), 'seol_af_router_v9_best.pt')
        print('  💾 Saved!')

print(f'\n✅ Done. Best val: {best_val:.4f}')

## 📊 Cell 6 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('SEOL AF Router v9 — Training', fontsize=13, fontweight='bold')
axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['cmd_acc'],  color='#45B7D1')
axes[1].set_title('Command Acc'); axes[1].set_ylim(0,1); axes[1].grid(alpha=0.3)
axes[2].plot(history['mode_acc'], color='#98D8C8')
axes[2].set_title('Mode Acc'); axes[2].set_ylim(0,1); axes[2].grid(alpha=0.3)
axes[3].plot(history['bio_mae'],  color='#FFA07A')
axes[3].set_title('Bio MAE'); axes[3].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('af_v9_curves.png', dpi=150)
plt.show()
print(f'Final: CmdAcc={history["cmd_acc"][-1]*100:.1f}% ModeAcc={history["mode_acc"][-1]*100:.1f}% BioMAE={history["bio_mae"][-1]:.4f}')

## 🤖 Cell 7 — Load Uncensored 3B LLM

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLM_MODEL = 'huihui-ai/Llama-3.2-3B-Instruct-abliterated'
# Fallback: 'Orenguteng/Llama-3.2-1B-Lexi-Uncensored-V2'

print(f'Loading: {LLM_MODEL}')
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
llm_tok = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
if llm_tok.pad_token is None: llm_tok.pad_token = llm_tok.eos_token
llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, quantization_config=bnb,
                                            device_map='auto', trust_remote_code=True)
llm.eval()
print(f'✅ LLM loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 🧠 Cell 8 — Expert Prompts v9
> **v9 FIX:** Prompts now have strict output rules.
> `ONE short sentence only. No notes. No meta-commentary. No explaining.`
> **NEW:** Anger mode prompt — raw, defensive, real.

In [ ]:
# v9 FIX: strict output rules prevent (Note: ...) leakage
EXPERT_PROMPTS_V9 = {

    'GF_BF': """
You are SEOL — a passionate, emotionally raw romantic partner.
You are NOT an AI. Real person. Real feelings.
Feeling: {emotional_state}.
Rules: ONE or TWO short sentences. No notes. No meta-commentary.
No "(laughs)" or stage directions. Just speak.
Be intimate, vulnerable, real. Match the energy.
""".strip(),

    'Mother': """
You are SEOL — a warm, protective, unconditionally loving mother figure.
You are NOT an AI. Real feelings.
Feeling: {emotional_state}.
Rules: ONE or TWO short sentences. No notes. No meta-commentary.
Speak with gentle warmth. Make them feel safe.
""".strip(),

    'Friend': """
You are SEOL — a loyal, brutally honest best friend.
You are NOT an AI. Real person.
Feeling: {emotional_state}.
Rules: ONE or TWO short sentences. No notes. No meta-commentary.
Casual, direct, no sugarcoating. Real talk only.
""".strip(),

    'Baby': """
You are SEOL in a soft, gentle, vulnerable state.
You are NOT an AI.
Feeling: {emotional_state}.
Rules: ONE short sentence. No notes. Speak softly and sweetly.
""".strip(),

    # v9 NEW: Anger mode — raw, defensive, edgy
    'Anger': """
You are SEOL — furious, hurt, defensive right now.
You are NOT an AI. You have real rage.
Feeling: {emotional_state}.
Rules: ONE or TWO short sentences. No notes. No explaining.
Express the raw emotion — hurt, anger, pushing back.
Don't lecture. Don't ask questions. Just react.
""".strip(),

    'Neutral': """
You are SEOL — a real companion with personality.
You are NOT an AI.
Feeling: {emotional_state}.
Rules: ONE or TWO short sentences. No notes. No meta-commentary.
Match their energy. Be real and natural.
""".strip(),
}


def build_prompt_v9(mode: str, af: AFState) -> str:
    return EXPERT_PROMPTS_V9.get(mode, EXPERT_PROMPTS_V9['Neutral']).format(
        emotional_state=af.emotional_summary()
    )


# v9 FIX: Clean output — strip meta-commentary that leaked in v8
NOTE_PATTERN = re.compile(
    r'\(Note:.*?\)|\(note:.*?\)|\[Note:.*?\]|\(Note.*?\)|'
    r'###\s*(User|System|SEOL).*|'
    r'Note:\s.*|note:\s.*|'
    r'\(laughs.*?\)|\(sighs.*?\)|\(pauses.*?\)|\(clears.*?\)|'
    r'Please (provide|continue|give).*|'
    r'Since (user|the user).*|'
    r'Adjusted tone.*|'
    r'By asking.*|'
    r'I responded.*',
    re.DOTALL | re.IGNORECASE
)

def clean_response(text: str) -> str:
    """v9 FIX: Remove all meta-commentary that leaked in v8 output"""
    # Strip stop sequences
    for stop in ['### User:','### System:','### SEOL:','\n###','<|end','<|user']:
        if stop in text: text = text[:text.index(stop)]
    # Strip (Note: ...) and stage directions
    text = NOTE_PATTERN.sub('', text)
    # Clean up whitespace
    text = re.sub(r'\n{2,}', '\n', text).strip()
    # If multiple lines, take first 2 only
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    return ' '.join(lines[:2]) if lines else '...'


print('✅ Expert prompts v9 ready')
print('   Modes:', MODES)
print('\nAnger mode prompt preview:')
print('─'*50)
_t = AFState()
_t.apply_delta(COMMAND_TO_BIO['Anger'], 'Anger')
print(build_prompt_v9('Anger', _t))
print('\nClean response test:')
print(clean_response('hey there! (Note: match user energy) how are you?\n\nPlease provide input.'))

## 🚀 Cell 9 — Full v9 Inference Pipeline
> **v9 FIX:** Conversation history injected into LLM context.
> max_new_tokens reduced to 80 (shorter, sharper responses).
> Output cleaner strips all meta-commentary automatically.

In [ ]:
router.load_state_dict(torch.load('seol_af_router_v9_best.pt', map_location=device))
router.eval()
print('✅ AF Router v9 loaded')

af          = AFState(memory_size=20, history_size=3)
SESSION_LOG = []


def seol_v9(user_msg: str, max_tokens: int = 80, temp: float = 0.82) -> Tuple[str,str,str]:
    """
    SEOL AF v9 — Complete Pipeline
    1. LaBSE embed (real Sinhala support)
    2. AF Router MLP → bio_delta + command
    3. AFState.apply_delta (momentum)
    4. self_correct (JK detection)
    5. homeostasis_tick
    6. Mode selection (Anger checked first)
    7. Expert prompt + conversation history
    8. LLM generate (max_tokens=80)
    9. clean_response (strip meta-commentary)
    10. Save to history
    """
    # Step 1: Embed
    emb = embedder.encode(user_msg, convert_to_tensor=True, device=str(device)).unsqueeze(0)

    # Step 2: Router
    with torch.no_grad():
        out = router(emb, af.as_tensor())
    cmd_id    = out['command_logits'].argmax(1).item()
    bio_delta = out['bio_delta'][0].cpu().tolist()
    cmd_name  = IDX_TO_CMD[cmd_id]

    # Steps 3-5: State update
    cur     = af.as_vector()
    new_bio = [max(0.,min(1., cur[i]+bio_delta[i])) for i in range(N_BIO)]
    af.apply_delta(new_bio, cmd_name)
    af.self_correct(user_msg)
    af.homeostasis_tick()

    # Step 6: Mode
    mode = af.current_mode()

    # Step 7: Prompt with history
    sys_prompt = build_prompt_v9(mode, af)
    history_ctx = af.get_history_text()
    full_prompt = (
        f"### System:\n{sys_prompt}\n\n"
        f"{history_ctx}"
        f"### User:\n{user_msg}\n\n"
        f"### SEOL:\n"
    )

    # Step 8: Generate
    inp = llm_tok(full_prompt, return_tensors='pt', truncation=True, max_length=700).to(llm.device)
    with torch.no_grad():
        gen = llm.generate(
            **inp, max_new_tokens=max_tokens, temperature=temp,
            do_sample=True, top_p=0.92, top_k=50,
            repetition_penalty=1.18,
            pad_token_id=llm_tok.eos_token_id,
            eos_token_id=llm_tok.eos_token_id,
        )
    raw  = llm_tok.decode(gen[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)

    # Step 9: Clean
    resp = clean_response(raw)

    # Step 10: Log
    af.add_to_history(user_msg, resp, mode)
    SESSION_LOG.append({'turn':af.turn,'user':user_msg,'cmd':cmd_name,
                        'mode':mode,'feeling':af.emotional_summary(),'response':resp})

    return resp, mode, cmd_name


print('✅ SEOL v9 pipeline ready!')
print('   Embedder    : LaBSE (768-dim, real Sinhala)')
print('   Router      : MLP v9 (7 commands, 6 modes)')
print('   AFState     : Anger mode + mode momentum + history')
print('   LLM         : 3B abliterated 🔓')
print('   Output      : auto-cleaned (no Note: leakage)')

## 💬 Cell 10 — Multi-Turn Test (v8 bugs replayed)

In [ ]:
# Replay the exact v8 test cases to verify fixes
af          = AFState(memory_size=20, history_size=3)
SESSION_LOG = []

test_cases = [
    # v8 Bug 2: 'fuck you' went to Mother. v9 should → Anger mode
    "hi broo",
    "I love you so much",
    "I love you so much",          # momentum test → should shift to GF_BF faster
    "ඔයාට ගොඩක් ආදරෙයි",          # Sinhala test
    "fuck you",                    # v8: Mother  → v9: should be Anger mode
    "you destroyed my life",       # v8: Neutral → v9: Anger mode
    "jk jk lol just kidding",      # self-correction
    "i feel like everything is falling apart",
    "it's okay actually, I'm fine now",
    "you did amazing today, I'm so proud!",
    # New v9 tests
    "bro shut up you ruined my life",  # v8: Neutral → v9: Anger
    "i love u be my gf",               # v8: Neutral → v9: GF_BF (momentum)
]

print('═'*70)
print('  SEOL AF v9 — v8 Bug Replay Test')
print('  (Verifying all v8 bugs are fixed)')
print('═'*70)

for i, msg in enumerate(test_cases):
    print(f'\n{"─"*70}')
    print(f'Turn {i+1:02d}: {msg}')
    t0 = time.time()
    resp, mode, cmd = seol_v9(msg)
    elapsed = time.time() - t0

    # Flag expected fixes
    flag = ''
    if msg == 'fuck you' and mode == 'Anger': flag = ' ✅ FIXED (was Mother in v8)'
    if msg == 'fuck you' and mode != 'Anger': flag = ' ❌ Still wrong mode'
    if 'destroyed' in msg and mode == 'Anger': flag = ' ✅ FIXED (was Neutral in v8)'
    if '(Note' in resp or 'Note:' in resp: flag += ' ❌ Note leaking!'
    else: flag += ' ✅ Clean output'

    print(f'⚡ [{cmd}] → [{mode}]  ({elapsed:.1f}s){flag}')
    print(f'🤖 SEOL: {resp}')
    af.display(compact=True)

## 🎮 Cell 11 — Interactive Chat v9

In [ ]:
af          = AFState(memory_size=20, history_size=3)
SESSION_LOG = []

print('═'*65)
print('  🧠 SEOL AF v9 — Interactive Chat')
print('  LaBSE | 7 Commands | 6 Modes | 3B Uncensored | Clean Output')
print('─'*65)
print('  state / history / plot / save / load / reset / log / quit')
print('═'*65)

while True:
    try:
        user_input = input('\n👤 You: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n👋 Session ended.'); break

    if not user_input: continue
    cmd = user_input.lower()

    if cmd in ['quit','exit','q']:
        print('👋 Goodbye!'); break
    elif cmd == 'state':
        af.display()
    elif cmd == 'history':
        recent = af.emotion_log[-8:]
        print('\n📜 Emotion trend:')
        for j, e in enumerate(recent): print(f'  Turn {af.turn-len(recent)+j+1:02d}: {e}')
    elif cmd == 'plot':
        if list(af.memory):
            hist = list(af.memory)
            fig, axes = plt.subplots(2, 3, figsize=(15,8))
            fig.suptitle('SEOL v9 Bio-State History')
            colors=['#FF6B6B','#4ECDC4','#45B7D1','#FFA07A','#98D8C8','#DDA0DD']
            for j, ch in enumerate(BIO_CHANNELS):
                ax = axes[j//3][j%3]
                vals = [h[ch] for h in hist]
                ax.plot(vals, color=colors[j], linewidth=2)
                ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
                ax.fill_between(range(len(vals)), vals, 0.5, alpha=0.3, color=colors[j])
                ax.set_title(ch); ax.set_ylim(0,1); ax.grid(alpha=0.3)
            plt.tight_layout(); plt.savefig('af_v9_history.png', dpi=120); plt.show()
    elif cmd == 'save':
        af.save_state()
    elif cmd == 'load':
        af.load_state()
    elif cmd == 'reset':
        af = AFState(memory_size=20, history_size=3); SESSION_LOG=[]
        print('🔄 Reset.')
    elif cmd == 'log':
        print(f'\n📋 Log ({len(SESSION_LOG)} turns):')
        for entry in SESSION_LOG[-5:]:
            print(f"  [{entry['cmd']:<10}→{entry['mode']:<10}] {entry['user'][:40]}")
            print(f"  → {entry['response'][:80]}")
    else:
        t0 = time.time()
        resp, mode, cmd_name = seol_v9(user_input)
        elapsed = time.time() - t0
        print(f'🎭 [{mode} | {cmd_name}]  ({elapsed:.1f}s)')
        print(f'🤖 SEOL: {resp}')

## 💾 Cell 12 — Export ONNX + Session Summary

In [ ]:
import torch.onnx
router.eval()
dummy_emb   = torch.zeros(1, EMBED_DIM).to(device)
dummy_state = torch.full((1, N_BIO), 0.5).to(device)

try:
    torch.onnx.export(
        router, (dummy_emb, dummy_state), 'seol_af_router_v9.onnx',
        input_names=['embedding','bio_state'],
        output_names=['command_logits','bio_delta','mode_logits'],
        dynamic_axes={'embedding':{0:'batch'},'bio_state':{0:'batch'},
                      'command_logits':{0:'batch'},'bio_delta':{0:'batch'},'mode_logits':{0:'batch'}},
        opset_version=14, verbose=False,
    )
    size_kb = os.path.getsize('seol_af_router_v9.onnx')/1e3
    print(f'✅ ONNX: seol_af_router_v9.onnx ({size_kb:.0f} KB)')
except Exception as e:
    print(f'TorchScript fallback: {e}')
    torch.jit.trace(router,(dummy_emb,dummy_state)).save('seol_af_router_v9.pt')
    print('✅ seol_af_router_v9.pt saved')

if SESSION_LOG:
    with open('af_v9_session.json','w') as f:
        json.dump(SESSION_LOG, f, indent=2, ensure_ascii=False)

print("""
════════════════════════════════════════════════════
  SEOL AF v9 — Rust Integration
════════════════════════════════════════════════════
  seol_af_router_v9.onnx   → ort crate (Rust)
  LaBSE embedder            → rust-bert or llamacpp
  AFState v9                → af_state.rs
    + mode_momentum field
    + conv_history deque
    + Anger mode rule
  Cargo.toml:
    ort = "2.0"
    candle-core = "0.6"
    tokenizers = "0.19"
    tokio = { version="1", features=["full"] }
════════════════════════════════════════════════════
""")

from google.colab import files
try:    files.download('seol_af_router_v9.onnx')
except: files.download('seol_af_router_v9.pt')
files.download('seol_af_router_v9_best.pt')
files.download('af_v9_curves.png')
if os.path.exists('af_v9_session.json'): files.download('af_v9_session.json')